# Notebook 03 — Credit Cluster Scoring and Reporting Tool

In the previous notebooks, I built and challenged the credit clustering model. The model was trained on public SEC EDGAR financial statement data, tested against alternative clustering methods, and saved as a reusable artifact. This notebook is where the project becomes practical.

The purpose of Notebook 03 is to apply the trained v3 KMeans model to a real, private, simulated, or competitor company. In other words, this is no longer only about understanding the historical SEC training universe. Here, I want to take a company’s financial statements, convert them into the same engineered credit-risk feature space, assign the company to one of the trained credit-risk clusters, and produce a structured business interpretation.

The scoring process follows the same logic used during model training. The input financials are transformed into ratios, component risk indicators, and the six domain-level model features. The saved KMeans artifact then assigns the company to the nearest trained centroid. After that, the notebook adds interpretation layers: cluster label, risk rank, scorecard credit score, distance diagnostics, soft affinity, near-default affinity, adjacent-bucket outlook, feature coverage, warning flags, and guardrails.

The main output is not just a number. The goal is to create a practical analyst-support workflow. The notebook produces an Excel workbook for detailed review and a professional PDF report for communication. The PDF report is especially important because the final result must be understandable to a finance or business user, not only to someone reading Python code.

The expected workflow is:

1. Load the saved model artifact from the training notebook.
2. Enter company financials manually or load them from a CSV/Excel file.
3. Run the scoring pipeline.
4. Review the assigned risk bucket, scorecard score, affinities, warning flags, and guardrails.
5. Generate the Excel and PDF outputs.
6. Interpret the result carefully, using the model as a structured credit-risk diagnostic rather than as a formal rating.

A key limitation remains unchanged: the output is **not a formal credit rating**, **not a lending decision**, and **not a calibrated probability of default**. It is a model-relative financial-risk benchmark against the public-company credit clusters created in the previous notebooks. The result can support professional credit analysis, but it does not replace professional judgment.

For detailed interpretation rules, I refer to the [model_interpretation.md file](../docs/model_interpretation.md). For the scoring and reporting workflow, I refer to the [scorer_report_methodology.md file](../docs/scorer_report_methodology.md). For robustness and future sensitivity testing, I refer to the [sensitivity_analysis.md file](../docs/sensitivity_analysis.md).


In [1]:
#Import your libraries here

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import sys

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

### 1. Load saved fitted model artifact

This first step loads the fitted model artifact produced by the training notebook. The artifact is the bridge between model development and practical scoring. It contains the trained KMeans pipeline, the feature list used during training, the cluster labels, the ordered risk ranks, the cluster profiles, and the interpretation metadata needed to score a new company consistently.

Before loading the artifact, I set the project root and import the reusable reporting, scoring, diagnostics, guardrail, and configuration utilities. This keeps Notebook 03 focused on application rather than rebuilding logic already implemented in the source-code modules.

After running Notebook 01, the model artifact is expected to be located in:

```python
outputs/saved_models/nonfinancial_credit_scorecard_kmeans_k5_v3.joblib
```

If the file is missing, the notebook raises a clear error. This is intentional because Notebook 03 depends on the trained artifact from the previous notebook and should not silently continue without the fitted model.

After loading the artifact, I identify the scoring segment, scoring temperature, FX conversion setting, and denominator threshold used for ratio construction. In the current project version, the primary scoring segment is `Non-financial`, because financial and real estate companies require separate feature engineering and are outside the scope of this model.

I also infer the near-default cluster from the artifact. This is important because the raw KMeans cluster IDs are arbitrary. The weakest cluster is not determined by its numeric cluster ID, but by the stored risk-rank mapping and profile interpretation created during the training and profiling stage.

In [2]:
# Import project modules and load trained artifact

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils import (
    build_credit_report_tables,
    save_credit_report_outputs,
    save_credit_pdf_report,
)

from src.credit_clustering import (
    # Artifact utilities
    load_artifact,
    get_segment_artifact,
    infer_near_default_cluster_from_artifact,

    # Scoring and diagnostics
    score_companies,
    add_adjacent_bucket_distances_and_outlook,
    compare_to_cluster_profile,
    make_scenarios,

    #Analyst guardrails
    apply_credit_guardrails,

    # Config
    DEFAULT_PRIMARY_SEGMENT,
    DEFAULT_SCORING_TEMPERATURE,
    DEFAULT_FX_TO_MODEL_CURRENCY,
    DEFAULT_SCORING_MIN_DENOMINATOR,
    RATIO_COLS,
    SUMMARY_COLS_WITH_OUTLOOK,
    SCENARIO_SUMMARY_COLS,
)

MODEL_PATH = (
    PROJECT_ROOT
    / "outputs"
    / "saved_models"
    / "nonfinancial_credit_scorecard_kmeans_k5_v3.joblib"
)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model artifact not found at {MODEL_PATH}")

INPUT_PATH = PROJECT_ROOT / "inputs"
INPUT_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = PROJECT_ROOT / "outputs" / "company_scores"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

artifact = load_artifact(MODEL_PATH)

SCORING_SEGMENT = artifact.get("primary_segment", DEFAULT_PRIMARY_SEGMENT)
SCORING_TEMPERATURE = DEFAULT_SCORING_TEMPERATURE
FX_TO_MODEL_CURRENCY = DEFAULT_FX_TO_MODEL_CURRENCY
SCORING_MIN_DENOMINATOR = DEFAULT_SCORING_MIN_DENOMINATOR

NEAR_DEFAULT_CLUSTER = infer_near_default_cluster_from_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

segment_artifact = get_segment_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("Scoring segment:", SCORING_SEGMENT)
print("Artifact version:", artifact.get("artifact_version"))
print("Features used by model:", segment_artifact.get("feature_cols"))
print("Cluster labels:", segment_artifact.get("cluster_labels"))
print("Risk rank:", segment_artifact.get("risk_rank"))
print("Inferred near-default cluster:", NEAR_DEFAULT_CLUSTER)

PROJECT_ROOT: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3
MODEL_PATH: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\saved_models\nonfinancial_credit_scorecard_kmeans_k5_v3.joblib
Scoring segment: Non-financial
Artifact version: v3_scorecard
Features used by model: ['earnings_risk', 'structural_distress_risk', 'operating_cashflow_risk', 'liquidity_risk', 'debt_service_risk', 'leverage_risk']
Cluster labels: {4: '1 - Strong relative credit profile', 1: '2 - Good credit profile', 2: '3 - Leveraged / elevated risk profile', 0: '4 - Weak credit profile', 3: '5 - Distressed / near-default proxy'}
Risk rank: {4: 1, 1: 2, 2: 3, 0: 4, 3: 5}
Inferred near-default cluster: 3


### 2. Manual input example

This step defines the company financial inputs that will be scored by the saved model artifact. The user can enter the financial statements manually in a Python dictionary, convert them into a one-row `DataFrame`, and then save the input as a CSV file. This makes the notebook flexible: the same scoring workflow can be used for a private company, a competitor, a simulated case, or a real public issuer where updated financials are available.

I show two separate examples. The first example is a manual company case based on the financial data of the company where I work as a finance manager. This company does not have an official external credit rating, which is exactly the type of situation where a model-relative benchmarking tool can be useful. The purpose is not to produce a formal rating, but to understand where the company’s financial profile falls relative to the public-company benchmark clusters.

The second example uses Eastern European Electric Company B.V. (`EEEC`), a Netherlands-based consolidated entity of the Bulgarian energy group Electrohold. This case is useful because EEEC has publicly disclosed external credit ratings, including `Ba2` by Moody’s and `BB` by Fitch, both with stable outlooks. This does not mean that my model should reproduce the agency ratings exactly. The model does not use official ratings as labels and is not calibrated to agency methodologies. However, EEEC is a useful real-world reference point because I can compare the model-relative output with a company that has an external market credit assessment.

The input structure contains the main financial statement values required for scoring: revenue, assets, liabilities, equity, current assets and liabilities, cash, receivables, inventory, debt, net income, operating cash flow, capex, operating income, interest expense, depreciation and amortization, and EBITDA where available. If direct EBITDA is missing, the scoring logic can reconstruct it from `operating_income + depreciation_amortization`, provided both inputs are supplied.

Currency and unit consistency are important. The first manual example is already converted from thousand BGN into full USD units using an assumed exchange rate. The EEEC example is entered in BGN. If the model currency is USD, the FX conversion setting should be adjusted consistently before scoring. Most ratios are scale-neutral, but `log_assets`, asset-size diagnostics, and some warning flags depend on the absolute size of the company.

At the end of this step, I select which CSV input file will be scored by setting `template_path` and `current_file_name`. This allows me to switch between cases without changing the scoring pipeline itself.


In [3]:
manual_input_2025 = pd.DataFrame([{
    "company_name": "Manual 2025 Company",
    "fiscal_year": 2025,
    "currency": "USD",
    "major_sector": "Manufacturing / Industrials",
    "financial_flag": "Non-financial",

    # Values converted from thousand BGN to full USD units at 1.7 BGN/USD
    "revenue": 48_585_294.12,
    "assets": 29_721_275.23,
    "current_assets": 10_037_745.82,
    "cash": 34_804.65,
    "receivables": 2_208_235.29,
    "inventory": 7_794_705.88,
    "equity": 9_082_352.94,
    "current_liabilities": 12_722_352.94,
    "liabilities": 20_638_823.53,
    "long_term_debt": 7_910_588.24,
    "short_term_debt": 1_478_235.29,
    "net_income": 949_411.76,
    "cfo": 3_558_235.29,
    "capex": 588_235.29,
    "operating_income": 1_664_705.88,
    "interest_expense": 597_647.06,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 2_713_000,
    "ebitda": np.nan,
}])

template_path = INPUT_PATH / "model_2025_company.csv"

manual_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)


Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\inputs\model_2025_company.csv


In [4]:
eeec_input_2025 = pd.DataFrame([{
    "company_name": "Eastern European Electric Company B.V.",
    "fiscal_year": 2025,
    "currency": "BGN",
    "major_sector": "Transportation / Utilities",
    "financial_flag": "Non-financial",
    "revenue": 2_572_207_000,
    "assets": 2_127_723_000,
    "current_assets": 1_024_267_000,
    "cash": 224_558_000,
    "receivables": 465_950_000,
    "inventory": 42_958_000,
    "equity": 457_686_000,
    "current_liabilities": 541_002_000,
    "liabilities": 1_670_037_000,
    "long_term_debt": 988_887_000,
    "short_term_debt": 14_753_000,
    "net_income": 36_799_000,
    "cfo": 265_267_000,
    "capex": 212_690_000,
    "operating_income": 54_961_000,
    "interest_expense": 102_302_000,

    # Optional EBITDA inputs used by the patched v3 scorer.
    # If direct EBITDA is not available, scoring.py can calculate it from
    # operating_income + depreciation_amortization when both are supplied.
    "depreciation_amortization": 119_519_000,
    "ebitda": 259_574_000,
}])

template_path = INPUT_PATH / "EEEC_2025.csv"

eeec_input_2025.to_csv(template_path, index=False)
print("Template saved to:", template_path)

Template saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\inputs\EEEC_2025.csv


In [5]:
#Configure input data model
template_path = INPUT_PATH / "EEEC_2025.csv"
current_input = pd.read_csv(template_path)

#Configure output files common name
current_file_name = "EEEC_2025"

display(current_input)

,company_name,fiscal_year,currency,major_sector,financial_flag,revenue,assets,current_assets,cash,receivables,inventory,equity,current_liabilities,liabilities,long_term_debt,short_term_debt,net_income,cfo,capex,operating_income,interest_expense,depreciation_amortization,ebitda
0,Eastern European Electric Company B.V.,2025,BGN,Transportation / Utilities,Non-financial,2572207000,2127723000,1024267000,224558000,465950000,42958000,457686000,541002000,1670037000,988887000,14753000,36799000,265267000,212690000,54961000,102302000,119519000,259574000


### 3. Score the selected company and add diagnostics

This step applies the trained model artifact to the selected company input. The main scoring logic lives in `src.credit_clustering.scoring.py`. This module intentionally does not rebuild the feature-engineering logic. Instead, it calls the same feature engineering process used during model training, so the company being scored is converted into the same six-domain credit-risk feature space as the SEC training observations.

The `score_companies()` function performs the core model application. It takes the raw company financials, applies FX conversion if required, calculates the derived accounting values, builds the financial ratios, transforms them into bounded risk features, and then uses the saved KMeans pipeline to assign the company to the nearest trained cluster centroid. It also calculates the distance to each cluster, converts those distances into soft cluster affinities, maps the raw cluster ID to the stored business label and risk rank, calculates feature coverage, and creates warning flags.

After the initial score is produced, I add the adjacent-bucket outlook using `add_adjacent_bucket_distances_and_outlook()`. This diagnostic compares the company’s distance to the immediately stronger and immediately weaker credit-risk buckets. The purpose is to understand whether the company is centered in its assigned bucket or whether its current financial profile is already leaning toward a neighboring bucket.

The logic is based on centroid distances. If the company is assigned to risk rank (r), the method looks at the centroid of the stronger adjacent bucket (r - 1), where available, and the centroid of the weaker adjacent bucket (r + 1), where available. In simplified form, the comparison is:

```text
distance to stronger adjacent bucket
vs.
distance to weaker adjacent bucket
```

The outlook is then classified as `Positive`, `Neutral`, or `Negative`. A `Positive` outlook means the company is closer to the stronger adjacent bucket than to the weaker one by a sufficient margin. A `Negative` outlook means the company is closer to the weaker adjacent bucket by a sufficient margin. A `Neutral` outlook means the company is not clearly pulled toward either neighboring bucket.

The notebook uses a `neutral_band` of `0.15` and boundary multipliers of `1.35` for both upgrade and downgrade direction. This is important because I do not want the model to overreact to very small differences in distance. A company should not receive a positive or negative outlook only because it is marginally closer to one adjacent centroid. The neutral band and boundary multipliers create a practical buffer zone around the assigned cluster.

Conceptually, the logic can be summarized as:

```text
if distance_to_stronger_bucket is materially lower than distance_to_weaker_bucket:
    outlook = Positive

elif distance_to_weaker_bucket is materially lower than distance_to_stronger_bucket:
    outlook = Negative

else:
    outlook = Neutral
```

This outlook is not a forecast and should not be interpreted as a rating migration prediction. It does not say that the company will improve or deteriorate in the future. It only describes the company’s current position inside the trained KMeans credit-risk space. In other words, it answers a practical analyst question: “Is this company comfortably inside its assigned bucket, or is it already sitting near the boundary with a better or worse neighboring bucket?”

I then apply the analyst guardrail layer with `apply_credit_guardrails()`. Guardrails are not part of the KMeans model itself. They are a professional review layer that checks for specific red flags such as high leverage, weak liquidity, poor coverage, or structural distress. This is important because a company may receive a reasonable cluster assignment while still having one or more financial issues that require caution in the final interpretation.

The output is shown in two parts. The first table displays the main scoring summary: company name, assigned cluster, business label, risk rank, cluster affinity, near-default affinity, distance diagnostics, outlook, feature coverage, warning flags, and guardrail interpretation. The second table shows the ratio and diagnostic snapshot in a transposed format, which makes it easier to review the company’s financial profile line by line.

This step is where the saved model becomes an actual scoring tool. It does not produce a formal credit rating, but it does produce a structured model-relative credit-risk assessment supported by distances, affinities, warning flags, and analyst guardrails.

In [6]:
scored_manual = score_companies(
    input_df=current_input,
    artifact=artifact,
    segment=SCORING_SEGMENT,
    temperature=SCORING_TEMPERATURE,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
    min_denominator=DEFAULT_SCORING_MIN_DENOMINATOR,
    near_default_cluster=NEAR_DEFAULT_CLUSTER,
)

scored_manual_with_outlook = add_adjacent_bucket_distances_and_outlook(
    scored_manual,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

current_external_rating = None # optional, none is not existant
scored_manual_with_outlook = apply_credit_guardrails(
    scored_manual_with_outlook,
    external_rating=current_external_rating
)

existing_summary_cols_with_outlook = [
    col for col in SUMMARY_COLS_WITH_OUTLOOK
    if col in scored_manual_with_outlook.columns
]

missing_summary_cols_with_outlook = [
    col for col in SUMMARY_COLS_WITH_OUTLOOK
    if col not in scored_manual_with_outlook.columns
]

if missing_summary_cols_with_outlook:
    print("Missing summary columns:", missing_summary_cols_with_outlook)

display(scored_manual_with_outlook[existing_summary_cols_with_outlook])

,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,Eastern European Electric Company B.V.,2025,2,3 - Leveraged / elevated risk profile,3,0.270951,0.186783,0.370699,1,2 - Good credit profile,0.636118,0,4 - Weak credit profile,0.799111,Neutral,Company remains materially closer to its assig...,59.843817,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


In [7]:
ratio_cols = artifact.get("ratio_cols", RATIO_COLS)

existing_ratio_cols = [
    col for col in ratio_cols
    if col in scored_manual.columns
]

missing_ratio_cols = [
    col for col in ratio_cols
    if col not in scored_manual.columns
]

if missing_ratio_cols:
    print("Missing ratio/diagnostic columns:", missing_ratio_cols)

company_ratio_snapshot = scored_manual[
    ["company_name"] + existing_ratio_cols
].copy()

display(company_ratio_snapshot.T.round(4))

,0
company_name,Eastern European Electric Company B.V.
log_assets,21.478318
liabilities_to_assets,0.784894
debt_to_assets,0.471697
debt_to_equity,2.192857
equity_to_assets,0.215106
cash_to_assets,0.105539
net_income_to_assets,0.017295
cfo_to_assets,0.124672
revenue_to_assets,1.208901


### Step 3 outcome review

For the rest of this notebook, I leave the scoring configuration set to `EEEC_2025`. This means the scoring pipeline is applied to Eastern European Electric Company B.V., the Netherlands-based consolidated entity of the Bulgarian energy group Electrohold.

The result assigns EEEC to cluster `2`, which corresponds to risk rank `3` and the label `3 - Leveraged / elevated risk profile`. This is a useful result because EEEC has external public credit ratings: `Ba2` by Moody’s and `BB` by Fitch, both with stable outlooks. These ratings are in the speculative-grade range, but they are not distressed ratings. In that sense, the model output is directionally consistent with the external rating profile: it does not classify the company as strong investment-grade-like, and it also does not classify it as near-default. It places it in the middle elevated-risk bucket.

The scorecard credit score is approximately `59.84`, which sits close to the upper end of the elevated-risk area and near the transition toward weaker credit profiles. The assigned-cluster affinity is approximately `0.27`, which is not very high. This means the company is not a perfectly centered representative of its assigned bucket. However, the outlook remains `Neutral`, because the company is still materially closer to its assigned cluster than to the adjacent stronger or weaker buckets.

The ratio snapshot explains the model result. EEEC has meaningful leverage: liabilities/assets are approximately `78.5%`, debt/assets are approximately `47.2%`, and debt/equity is approximately `2.19x`. Debt/EBITDA is approximately `3.87x`, and net debt/EBITDA is approximately `3.00x`. These are not distressed levels, but they clearly point to a leveraged capital structure.

Liquidity is more balanced. The current ratio is approximately `1.89x`, and the quick ratio is approximately `1.28x`, which are acceptable. Cash/assets are approximately `10.6%`, which also provides some liquidity buffer. This is why the model does not treat the company as a pure liquidity-distress case.

The weakest part of the profile is debt service. EBIT interest coverage is approximately `0.54x`, meaning operating income does not fully cover interest expense. EBITDA interest coverage is stronger at approximately `2.54x`, but still not high enough to eliminate debt-service concern. This explains the `interest_coverage_below_1` warning flag and the `High caution` guardrail level. In other words, the model’s elevated-risk label is mainly supported by leverage and coverage pressure, not by a liquidity collapse.

The model also calculates a near-default affinity of approximately `0.19`. This should not be interpreted as a probability of default. It only means that the company has some similarity to the weakest model cluster, mainly because of leverage and coverage metrics. The value is not dominant, so the model does not classify the company as distressed.

Overall, this is a good practical test of the scoring workflow. The model assigns EEEC to a leveraged/elevated-risk bucket, which broadly corresponds to the company’s actual speculative-grade credit profile. At the same time, the model is more mechanical and financial-ratio-driven than Moody’s or Fitch. Rating agencies also consider business profile, regulatory environment, market position, ownership structure, access to refinancing, bond structure, and qualitative factors. Therefore, the result should be interpreted as a model-relative financial-risk benchmark, not as a replication of the official Ba2/BB ratings.


### Additional methodology explanation to near-default affinity and its interpretation

The model also calculates a near-default affinity of approximately `0.19`. This should not be interpreted as a probability of default. It is a normalized similarity measure based on the company’s distance to the weakest trained cluster centroid compared with its distance to all other cluster centroids. The general interpretation of distance, affinity, and near-default affinity is discussed in [model_interpretation.md](../docs/model_interpretation.md), while the mathematical scoring logic is also summarized in [scorer_report_methodology.md](../docs/scorer_report_methodology.md).

The model first calculates Euclidean distances from the scored company to each KMeans centroid in the six-dimensional credit-risk feature space:

$$
d_j = \lVert x - c_j \rVert_2
$$

where (x) is the company’s six-feature risk vector and (c_j) is the centroid of cluster (j). These distances are then converted into soft affinities using an exponential distance transformation:

$$
a_j = \frac{e^{-d_j / T}}{\sum_{i=1}^{k} e^{-d_i / T}}
$$

where (a_j) is the affinity to cluster (j), (d_j) is the distance to cluster (j), (T) is the temperature parameter, and (k) is the number of clusters. In this project, (k = 5), and the default temperature is (T = 1.0). The effect of the temperature parameter is also discussed in [sensitivity_analysis.md](../docs/sensitivity_analysis.md), because temperature affects affinities but not the actual cluster assignment.

The near-default affinity is simply the affinity assigned to the weakest cluster:

$$
a_{\text{near-default}} =
\frac{e^{-d_{\text{near-default}} / T}}
{\sum_{i=1}^{5} e^{-d_i / T}}
$$

The value always falls between `0` and `1`, and all cluster affinities for one company sum to `1.0`. A higher value means that the company is more similar to the weakest cluster centroid. A lower value means that the company is farther away from the weakest cluster compared with the other cluster centroids.

For practical interpretation, a near-default affinity below approximately `0.10` can be read as low proximity to the weakest cluster. A value around `0.10–0.25` suggests some similarity to the weakest cluster, but not a dominant signal. Values above `0.25–0.35` deserve closer review, especially if they are combined with weak guardrails or poor coverage. A value above `0.50` would mean that the weakest cluster is becoming one of the dominant similarity anchors for the company’s profile.

In EEEC’s case, the near-default affinity of approximately `0.19` indicates that the company has some measurable similarity to the weakest credit-risk profile, mainly because of leverage and debt-service pressure. However, the assigned-cluster affinity is higher, and the company is classified in the `Leveraged / elevated risk profile` bucket rather than the distressed or near-default proxy bucket. Therefore, the result should be read as an elevated-risk signal with some distress-like features, not as a default probability or a distressed classification.

### 4. Compare the company to cluster medians

This step compares the scored company to the median profile of its assigned cluster. The purpose is to explain why the company was assigned to a specific credit-risk bucket and to understand whether it looks typical or atypical for that bucket.

The function `compare_to_cluster_profile()` takes the scored company row and the saved model artifact. It then retrieves the median financial profile of the assigned cluster and compares the company’s own ratios against those cluster medians. The output is not used to change the cluster assignment. It is an interpretation table.

The table has four main columns:

* `company_value` shows the company’s own value for each ratio or diagnostic metric;
* `assigned_cluster_median` shows the median value of the same metric inside the assigned cluster;
* `difference` shows the numerical difference between the company and the cluster median;
* `relative_position` shows whether the company is above, below, or equal to the assigned-cluster median.

This comparison should be read carefully because “above” or “below” is not always good or bad by itself. For example, being above the cluster median for `current_ratio`, `quick_ratio`, `cash_to_assets`, or `cfo_to_assets` is generally positive. Being above the cluster median for `liabilities_to_assets` or `debt_to_assets` usually indicates higher leverage and therefore higher risk. Similarly, being below the cluster median for `interest_coverage`, `operating_margin`, or `ebitda_interest_coverage` usually points to weaker debt-service capacity or profitability.

The table is therefore best used as an analyst explanation tool. It helps identify which parts of the company’s profile support the assigned cluster and which parts look stronger or weaker than the typical company in that same bucket. A company can belong to a leveraged or elevated-risk cluster while still having above-median liquidity, or it can have acceptable leverage but below-median coverage and profitability. This is exactly why the cluster assignment should be interpreted together with the ratio comparison, warning flags, and guardrails.

In general, the most important reading is not one individual difference, but the overall pattern. If several leverage, liquidity, profitability, cash-flow, and coverage metrics point in the same direction, the cluster assignment is easier to explain. If the company is much stronger on some dimensions and much weaker on others, then the model output should be read as a mixed profile and the analyst narrative should explain those trade-offs.

In [8]:
comparison_to_cluster = compare_to_cluster_profile(
    scored_manual.iloc[0],
    artifact,
)

comparison_to_cluster

,company_value,assigned_cluster_median,difference,relative_position
metric,,,,
log_assets,21.4783,22.5007,-1.0224,below_cluster_median
liabilities_to_assets,0.7849,0.6999,0.0850,above_cluster_median
debt_to_assets,0.4717,0.3701,0.1016,above_cluster_median
equity_to_assets,0.2151,0.2781,-0.0630,below_cluster_median
current_ratio,1.8933,1.1176,0.7757,above_cluster_median
quick_ratio,1.2764,0.4643,0.8121,above_cluster_median
cash_to_assets,0.1055,0.0362,0.0693,above_cluster_median
net_income_to_assets,0.0173,0.0247,-0.0074,below_cluster_median
cfo_to_assets,0.1247,0.0649,0.0598,above_cluster_median


### 5. Scenario analysis

This section tests how the scored company behaves under several mechanical stress scenarios. The purpose is not to forecast the future. The purpose is to understand whether the company’s current credit-risk profile is robust, or whether relatively simple changes in financial assumptions can move it toward a weaker cluster.

The first step is to create scenario inputs using `make_scenarios()`. This function starts from the base company input and creates alternative versions of the same company under different stress cases. These scenarios adjust selected financial statement values such as revenue, profit, operating cash flow, cash, debt, liabilities, operating income, interest expense, depreciation and amortization, and EBITDA. The resulting table allows me to inspect the raw scenario assumptions before scoring them.

This is important because scenario analysis should not be treated as a black box. Before running the model, I want to see what financial inputs are actually changing. If the scenario assumptions are too mild, the score may not move. If they are too aggressive, the scenario may become unrealistic. Showing the scenario input table makes the stress design transparent.

The scenario analysis uses five cases: one base case and four mechanical stress cases. The base case is the company’s reported or manually entered financial profile. The stress cases then change selected financial statement inputs to test how sensitive the model output is to weaker operating performance, higher debt, liquidity pressure, or a combined near-default-style deterioration.

| Scenario              | What the scenario represents                                                                              | Main variables moving down vs base                                                     | Main variables moving up vs base                                                                                      |
| --------------------- | --------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------- |
| `base`                | Original company financials used as the reference case.                                                   | None.                                                                                  | None.                                                                                                                 |
| `revenue_down_15pct`  | Operating stress case where sales and related profitability/cash-flow metrics weaken.                     | Revenue, net income, operating income, CFO, EBITDA if linked to operating performance. | Risk metrics related to profitability, cash flow, and debt service may increase.                                      |
| `debt_up_25pct`       | Financing stress case where the company takes on more debt.                                               | Equity cushion may weaken indirectly through leverage ratios.                          | Long-term debt, short-term debt, total debt, liabilities, debt/assets, debt/EBITDA, net debt/EBITDA, leverage risk.   |
| `cash_burn_case`      | Liquidity and operating weakness case where the company consumes cash and financial flexibility declines. | Cash, CFO, net income, operating income, liquidity ratios, cash/assets.                | Liquidity risk, earnings risk, operating cash-flow risk, warning flags may increase.                                  |
| `near_default_stress` | Severe combined stress case designed to push the financial profile toward the weakest model bucket.       | Cash, equity, profitability, operating cash flow, coverage, EBITDA quality.            | Liabilities, leverage ratios, structural distress risk, debt-service risk, near-default affinity, guardrail severity. |

The goal is not to forecast these exact cases. They are controlled model sensitivities. Each scenario asks a practical question: if this financial pressure appeared in the company’s numbers, would the model still keep the company in the same bucket, or would it migrate toward a weaker credit-risk profile?

The most important comparison is against the `base` case. If a scenario changes the scorecard credit score, outlook, guardrail level, or assigned cluster, the movement should be explained through the financial variables that changed. This makes the scenario analysis useful not only as a model output, but also as a credit-analysis discussion.


After the scenario inputs are created, I score all scenarios using the same `score_companies()` function that was used for the base case. This keeps the process consistent. Each scenario is transformed into the same financial ratios, bounded risk features, KMeans cluster assignment, distance diagnostics, affinity measures, warning flags, and risk labels.

I then add the adjacent-bucket outlook and analyst guardrails to each scenario. This is useful because a scenario may not always move the company into a different cluster, but it may still weaken the company’s position inside the assigned cluster. For example, the company may remain in the same risk bucket but move closer to the weaker adjacent bucket or trigger stronger guardrail warnings.

The final scenario table combines the main scoring outputs with selected diagnostic ratios. This allows me to compare not only the assigned cluster and risk label, but also the financial reasons behind the movement. Key ratios such as liabilities/assets, debt/assets, debt/equity, equity/assets, current ratio, quick ratio, interest coverage, FCF/debt, EBITDA margin, debt/EBITDA, net debt/EBITDA, EBITDA interest coverage, and debt-service risk help explain why a scenario becomes stronger or weaker.

The value of this step is that it turns the model from a static scoring tool into a sensitivity tool. I can see whether the company is stable under stress, whether it quickly migrates toward weaker credit-risk buckets, or whether the model result is mainly driven by one specific financial pressure point such as leverage, liquidity, cash flow, or debt service.

The scenario results should be interpreted as mechanical sensitivities, not predictions. A migration to a weaker cluster under a stress case does not mean the company will deteriorate in reality. It only means that, under the stated financial assumptions, the company’s model-relative credit-risk profile becomes closer to a weaker trained cluster.


In [9]:
scenario_input = make_scenarios(current_input.iloc[0])

scenario_input_cols = [
    "scenario",
    "assets",
    "liabilities",
    "equity",
    "cash",
    "revenue",
    "net_income",
    "cfo",
    "long_term_debt",
    "short_term_debt",
    "operating_income",
    "interest_expense",
    "depreciation_amortization",
    "ebitda",
]

existing_scenario_input_cols = [
    col for col in scenario_input_cols
    if col in scenario_input.columns
]

missing_scenario_input_cols = [
    col for col in scenario_input_cols
    if col not in scenario_input.columns
]

if missing_scenario_input_cols:
    print("Missing scenario input columns:", missing_scenario_input_cols)

display(scenario_input[existing_scenario_input_cols])

,scenario,assets,liabilities,equity,cash,revenue,net_income,cfo,long_term_debt,short_term_debt,operating_income,interest_expense,depreciation_amortization,ebitda
0,base,2127723000,1.670037e+09,457686000.0,224558000.0,2.572207e+09,36799000.0,265267000.0,9.888870e+08,14753000.0,54961000.0,102302000.0,119519000,259574000
0,revenue_down_15pct,2127723000,1.670037e+09,457686000.0,224558000.0,2.186376e+09,25759300.0,198950250.0,9.888870e+08,14753000.0,38472700.0,102302000.0,119519000,259574000
0,debt_up_25pct,2127723000,1.917259e+09,457686000.0,175113650.0,2.572207e+09,36799000.0,265267000.0,1.236109e+09,14753000.0,54961000.0,102302000.0,119519000,259574000
0,cash_burn_case,2127723000,1.670037e+09,457686000.0,112279000.0,2.572207e+09,-36799000.0,-265267000.0,9.888870e+08,14753000.0,54961000.0,102302000.0,119519000,259574000
0,near_default_stress,2127723000,2.340495e+09,-212772300.0,224558000.0,2.572207e+09,36799000.0,265267000.0,1.595792e+09,319158450.0,40920800.0,102302000.0,119519000,259574000


In [10]:
scored_scenarios = score_companies(
    input_df=scenario_input,
    artifact=artifact,
    segment=SCORING_SEGMENT,
    temperature=SCORING_TEMPERATURE,
    fx_to_model_currency=FX_TO_MODEL_CURRENCY,
    min_denominator=SCORING_MIN_DENOMINATOR,
    near_default_cluster=NEAR_DEFAULT_CLUSTER,
)

scored_scenarios = add_adjacent_bucket_distances_and_outlook(
    scored_scenarios,
    artifact,
    neutral_band=0.15,
    upgrade_boundary_multiplier=1.35,
    downgrade_boundary_multiplier=1.35,
)

scored_scenarios = apply_credit_guardrails(
    scored_scenarios,
    external_rating=current_external_rating,
)

scenario_summary_cols = artifact.get("scenario_summary_cols", SCENARIO_SUMMARY_COLS)

# Add selected diagnostic ratios for direct scenario comparison.
scenario_diagnostic_cols = [
    "liabilities_to_assets",
    "debt_to_assets",
    "debt_to_equity",
    "equity_to_assets",
    "cfo_to_assets",
    "current_ratio",
    "quick_ratio",
    "interest_coverage",
    "fcf_to_debt",
    "ebitda",
    "ebitda_margin",
    "debt_to_ebitda",
    "net_debt_to_ebitda",
    "ebitda_interest_coverage",
    "debt_service_risk",
]

scenario_display_cols = []

for col in list(scenario_summary_cols) + scenario_diagnostic_cols:
    if col in scored_scenarios.columns and col not in scenario_display_cols:
        scenario_display_cols.append(col)

missing_scenario_cols = [
    col for col in list(scenario_summary_cols) + scenario_diagnostic_cols
    if col not in scored_scenarios.columns
]

if missing_scenario_cols:
    print("Missing scenario columns:", missing_scenario_cols)

display(scored_scenarios[scenario_display_cols])

,scenario,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cfo_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,ebitda,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,debt_service_risk
0,base,2,3 - Leveraged / elevated risk profile,3,0.270951,0.186783,0.370699,1,2 - Good credit profile,0.636118,0.0,4 - Weak credit profile,0.799111,Neutral,Company remains materially closer to its assig...,59.843817,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.784894,0.471697,2.192857,0.215106,0.124672,1.893278,1.276350,0.537243,0.052386,259574000.0,0.100915,3.866489,3.001387,2.537331,0.804994
1,revenue_down_15pct,2,3 - Leveraged / elevated risk profile,3,0.272800,0.203220,0.341096,1,2 - Good credit profile,0.688495,0.0,4 - Weak credit profile,0.715566,Neutral,Company remains materially closer to its assig...,63.849901,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.784894,0.471697,2.192857,0.215106,0.093504,1.893278,1.276350,0.376070,-0.013690,259574000.0,0.118723,3.866489,3.001387,2.537331,0.838033
2,debt_up_25pct,2,3 - Leveraged / elevated risk profile,3,0.276559,0.205779,0.442779,1,2 - Good credit profile,0.817299,0.0,4 - Weak credit profile,0.882260,Neutral,Company remains materially closer to its assig...,67.168450,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",0.901085,0.587887,2.733013,0.215106,0.124672,1.893278,1.184956,0.537243,0.042033,259574000.0,0.100915,4.818902,4.144283,2.537331,0.857792
3,cash_burn_case,3,5 - Distressed / near-default proxy,5,0.289125,0.289125,0.347624,0,4 - Weak credit profile,0.577161,NaN,None,NaN,Neutral,Only the stronger adjacent bucket is available...,78.678130,1.0,"interest_coverage_below_1, negative_cfo_to_assets",Override required,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to the weakest model-r...,The cluster assignment appears primarily suppo...,Manual analyst override is required. The model...,0.784894,0.471697,2.192857,0.215106,-0.124672,1.893278,1.068811,0.537243,-0.476224,259574000.0,0.100915,3.866489,3.433938,2.537331,0.881188
4,near_default_stress,2,3 - Leveraged / elevated risk profile,3,0.265066,0.231799,0.721877,1,2 - Good credit profile,1.124644,0.0,4 - Weak credit profile,1.108510,Neutral,Company remains materially closer to its assig...,75.954216,1.0,"liabilities_exceed_assets, negative_equity, hi...",Override required,"elevated_debt_to_assets, high_debt_to_assets, ...",Structural balance-sheet distress indicators w...,The cluster assignment appears primarily suppo...,Manual analyst override is required. The model...,1.100000,0.900000,-9.000000,-0.100000,0.124672,1.893278,1.276350,0.400000,0.027456,259574000.0,0.100915,7.377282,6.512180,2.537331,0.924135


### Scenario analysis outcome review

The scenario analysis gives a useful first view of how the EEEC 2025 profile behaves under mechanical stress cases. The base case assigns the company to risk rank `3`, labelled `Leveraged / elevated risk profile`, with a scorecard credit score of approximately `59.84`. This is the reference point for the scenario comparison.

The `revenue_down_15pct` case keeps the company in the same risk bucket, but the scorecard credit score increases from approximately `59.84` to `63.85`. This means the model sees a weaker profile, even though the cluster assignment does not change. The main deterioration comes from weaker operating cash flow, lower interest coverage, and negative free cash flow relative to debt. The result is useful because it shows that the company is sensitive to operating performance, but the stress is not strong enough to push it into a weaker cluster.

The `debt_up_25pct` case also keeps the company in the same risk bucket, but the scorecard credit score increases further to approximately `67.17`. The main pressure comes from higher liabilities/assets, higher debt/assets, higher debt/equity, weaker quick liquidity, and worse debt/EBITDA and net debt/EBITDA. This scenario confirms that leverage is one of the key drivers of the model’s risk interpretation for EEEC.

The `cash_burn_case` produces the clearest migration. The company moves from risk rank `3` to risk rank `5`, labelled `Distressed / near-default proxy`. The scorecard credit score increases to approximately `78.68`, and the guardrail level becomes `Override required`. This happens because operating cash flow turns negative, free cash flow to debt becomes strongly negative, and warning flags include both weak interest coverage and negative operating cash flow. In this case, the model treats the profile as much closer to the weakest cluster, even though the balance-sheet leverage ratios do not change dramatically. This is a good example of how cash-flow deterioration can dominate the credit interpretation.

The `near_default_stress` case is more complex. It produces severe balance-sheet red flags: liabilities exceed assets, equity turns negative, debt/assets increases to `0.90`, debt/EBITDA rises to approximately `7.38x`, and net debt/EBITDA rises to approximately `6.51x`. The scorecard credit score increases to approximately `75.95`, and the guardrail level is again `Override required`. Interestingly, the assigned cluster remains risk rank `3`, even though the warning flags and guardrails clearly indicate severe distress. This is exactly why the guardrail layer is important. The model assignment alone should not dominate the interpretation when structural balance-sheet distress indicators are triggered.

Overall, the scenario results show that the model is responsive to different types of stress. Operating weakness, leverage increase, cash burn, and balance-sheet deterioration affect the scorecard and diagnostics in different ways. The most important output is not only whether the assigned cluster changes, but also how the scorecard score, near-default affinity, outlook, warning flags, and guardrail level move together.

This is a basic scenario-analysis example. It is intentionally mechanical and easy to follow. A more developed future version could include richer scenario design, such as macroeconomic shocks, interest-rate stress, refinancing stress, regulated-utility-specific assumptions, working-capital shocks, margin compression, multi-year projections, or covenant-style diagnostics. For the current project, however, this section already demonstrates that the trained clustering model can be used not only for a single static score, but also for structured sensitivity analysis around a company’s financial profile.


### 6. Save report outputs to files

This final step saves the scoring results into reusable report files. Up to this point, the notebook has produced several separate outputs: the base company score, the ratio snapshot, the comparison to the assigned cluster median, the scenario analysis, and the guardrail interpretation. In this step, these outputs are collected into a structured reporting package.

The function `build_credit_report_tables()` prepares the main tables needed for reporting. It combines the scored company result, the diagnostic ratio snapshot, the cluster comparison table, the scenario input assumptions, the scored scenario results, and the model artifact metadata. This creates one consistent reporting structure that can be exported into files instead of remaining only as notebook output.

The function `save_credit_report_outputs()` then saves the results into CSV and Excel files. The CSV files are useful for simple downstream review or further processing. The Excel workbook is the main analyst-facing file because it collects the score summary, scenario analysis, ratio diagnostics, cluster comparison, guardrails, and model-related outputs in one place.

The notebook also displays the key saved tables after export. This gives a quick visual confirmation that the files contain the expected scoring results. I also display the guardrail columns separately because they are an important part of the professional interpretation layer. The guardrails explain whether the model output should be accepted normally, qualified with caution, or overridden by manual analyst judgment.

Finally, `save_credit_pdf_report()` creates a professional PDF credit-risk report. This is the communication output of the notebook. The PDF is designed to summarize the model-relative risk label, scorecard result, key diagnostics, guardrails, cluster comparison, scenario analysis, and limitations in a format that can be shared more easily than a raw notebook or Excel file.

The outcome of this step is that Notebook 03 no longer ends with temporary Python tables only. It produces a complete scoring package:

| Output         | Purpose                                                 |
| -------------- | ------------------------------------------------------- |
| Score CSV      | Compact base-case scoring output.                       |
| Scenario CSV   | Scenario scoring results for sensitivity review.        |
| Excel workbook | Detailed analyst workbook with tables and diagnostics.  |
| PDF report     | Professional summary report for business communication. |

This is important for the project because it turns the trained model into a usable business tool. The notebook can now score a company, explain the result, test scenarios, apply guardrails, and export the output into files that are suitable for review, documentation, and presentation.

In [11]:
report_tables = build_credit_report_tables(
    scored_manual_with_outlook=scored_manual_with_outlook,
    scored_manual=scored_manual,
    comparison_to_cluster=comparison_to_cluster,
    scenario_input=scenario_input,
    scored_scenarios=scored_scenarios,
    artifact=artifact,
)

saved_paths = save_credit_report_outputs(
    tables=report_tables,
    output_path=OUTPUT_PATH,
    base_filename=current_file_name,
)

print("Scores saved to:", saved_paths["score_csv"])
print("Scenario CSV saved to:", saved_paths["scenario_csv"])
print("Full Excel score report saved to:", saved_paths["report_xlsx"])

display(report_tables["scored_file"])
display(report_tables["scenario_file"])

Scores saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_score_result.csv
Scenario CSV saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_scenario_analysis.csv
Full Excel score report saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_score_report.xlsx


,company_name,fiscal_year,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,Eastern European Electric Company B.V.,2025,2,3 - Leveraged / elevated risk profile,3,0.270951,0.186783,0.370699,1,2 - Good credit profile,0.636118,0,4 - Weak credit profile,0.799111,Neutral,Company remains materially closer to its assig...,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


,scenario,assigned_cluster,cluster_label,risk_rank,cluster_affinity,near_default_affinity,distance_to_assigned_cluster,upper_bucket_cluster,upper_bucket_label,distance_to_upper_bucket,lower_bucket_cluster,lower_bucket_label,distance_to_lower_bucket,outlook_flag,outlook_reason,scorecard_credit_score,feature_coverage_pct,warning_flags,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion,log_assets,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cash_to_assets,net_income_to_assets,cfo_to_assets,revenue_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,operating_margin,gross_margin,cfo_to_liabilities,capex_to_revenue,total_debt,net_debt,ebitda,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,leverage_risk,liquidity_risk,earnings_risk,operating_cashflow_risk,debt_service_risk,debt_service_risk_legacy,structural_distress_risk
0,base,2,3 - Leveraged / elevated risk profile,3,0.270951,0.186783,0.370699,1,2 - Good credit profile,0.636118,0.0,4 - Weak credit profile,0.799111,Neutral,Company remains materially closer to its assig...,59.843817,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",21.478318,0.784894,0.471697,2.192857,0.215106,0.105539,0.017295,0.124672,1.208901,1.893278,1.276350,0.537243,0.052386,0.021367,NaN,0.158839,0.082688,1.003640e+09,7.790820e+08,259574000.0,0.100915,3.866489,3.001387,2.537331,0.625658,0.350685,0.551367,0.440049,0.804994,0.878091,0.699837
1,revenue_down_15pct,2,3 - Leveraged / elevated risk profile,3,0.272800,0.203220,0.341096,1,2 - Good credit profile,0.688495,0.0,4 - Weak credit profile,0.715566,Neutral,Company remains materially closer to its assig...,63.849901,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",21.478318,0.784894,0.471697,2.192857,0.215106,0.105539,0.012107,0.093504,1.027566,1.893278,1.276350,0.376070,-0.013690,0.017597,NaN,0.119129,0.097280,1.003640e+09,7.790820e+08,259574000.0,0.118723,3.866489,3.001387,2.537331,0.625658,0.378217,0.585957,0.559347,0.838033,0.930952,0.699837
2,debt_up_25pct,2,3 - Leveraged / elevated risk profile,3,0.276559,0.205779,0.442779,1,2 - Good credit profile,0.817299,0.0,4 - Weak credit profile,0.882260,Neutral,Company remains materially closer to its assig...,67.168450,1.0,interest_coverage_below_1,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ...",21.478318,0.901085,0.587887,2.733013,0.215106,0.082301,0.017295,0.124672,1.208901,1.893278,1.184956,0.537243,0.042033,0.021367,NaN,0.138357,0.082688,1.250862e+09,1.075748e+09,259574000.0,0.100915,4.818902,4.144283,2.537331,0.795404,0.400973,0.551367,0.485081,0.857792,0.886374,0.771339
3,cash_burn_case,3,5 - Distressed / near-default proxy,5,0.289125,0.289125,0.347624,0,4 - Weak credit profile,0.577161,NaN,None,NaN,Neutral,Only the stronger adjacent bucket is available...,78.678130,1.0,"interest_coverage_below_1, negative_cfo_to_assets",Override required,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to the weakest model-r...,The cluster assignment appears primarily suppo...,Manual analyst override is required. The model...,21.478318,0.784894,0.471697,2.192857,0.215106,0.052770,-0.017295,-0.124672,1.208901,1.893278,1.068811,0.537243,-0.476224,0.021367,NaN,-0.158839,0.082688,1.003640e+09,8.913610e+08,259574000.0,0.100915,3.866489,3.433938,2.537331,0.652692,0.510245,0.781967,1.000000,0.881188,1.000000,0.699837
4,near_default_stress,2,3 - 

In [12]:
guardrail_cols = [
    "guardrail_level",
    "guardrail_flags",
    "guardrail_summary",
    "analyst_interpretation",
    "commercial_conclusion",
]

display(scored_manual_with_outlook[guardrail_cols])

,guardrail_level,guardrail_flags,guardrail_summary,analyst_interpretation,commercial_conclusion
0,High caution,"elevated_debt_to_assets, elevated_debt_to_ebit...",The company is assigned to 3 - Leveraged / ele...,The cluster assignment appears primarily suppo...,"The model output may be directionally useful, ..."


In [13]:
pdf_path = save_credit_pdf_report(
    report_tables=report_tables,
    scored_manual_with_outlook=scored_manual_with_outlook,
    comparison_to_cluster=comparison_to_cluster,
    scored_scenarios=scored_scenarios,
    artifact=artifact,
    output_path=OUTPUT_PATH,
    base_filename=current_file_name,
)

print("PDF credit report saved to:", pdf_path)

PDF credit report saved to: D:\users\kamen.dimitrov\desktop\SOFTUNI\AI_and_ML_upskill_program\machine_learning\08_final_project_3\outputs\company_scores\EEEC_2025_credit_report.pdf


### 7. Optional: inspect saved artifact internals

This final step is optional because it does not change the scoring result, create new model outputs, or affect the generated Excel and PDF reports. The actual scoring workflow is already complete at this point. However, inspecting the saved artifact is useful for transparency, debugging, and model governance.

The artifact is the object that connects the training notebook with the scoring notebook. It contains the fitted KMeans pipeline, the model feature list, the primary scoring segment, cluster labels, risk-rank mappings, cluster sizes, cluster profiles, and ranked cluster interpretation tables. By displaying these objects, I can confirm that Notebook 03 is using the correct trained model and the correct interpretation layer.

This is especially important because raw KMeans cluster IDs are arbitrary. The business meaning comes from the stored `cluster_labels` and `risk_rank` mappings. Inspecting the artifact confirms that the scoring output is being mapped correctly from technical cluster IDs to the ordered credit-risk labels.

The cluster sizes and cluster profile tables are also useful for review. They show how many issuer-year observations were used in each cluster and what the median financial profile of each cluster looks like. This helps verify that the scored company is being compared against the expected benchmark structure.

I keep this as Step 7 rather than part of the main scoring flow because it is mainly a control and inspection step. A normal user of the scoring notebook may not need to review the internals every time. However, for a final project, model review, or future development work, this section is helpful because it makes the saved artifact transparent and auditable.

In [14]:
# ---------------------------------------------------------------------
# Optional artifact inspection
# ---------------------------------------------------------------------

nonfin_artifact = get_segment_artifact(
    artifact,
    segment=SCORING_SEGMENT,
)

print("Artifact version:")
print(artifact.get("artifact_version"))

print("\nPrimary segment:")
print(artifact.get("primary_segment"))

print("\nFeatures used by model:")
print(nonfin_artifact.get("feature_cols"))

print("\nCluster labels:")
print(nonfin_artifact.get("cluster_labels"))

print("\nRisk rank:")
print(nonfin_artifact.get("risk_rank"))

print("\nCluster sizes:")
display(nonfin_artifact.get("cluster_sizes"))

print("\nCluster profile:")
display(nonfin_artifact.get("cluster_profile"))

print("\nCluster profile ranked:")
display(nonfin_artifact.get("cluster_profile_ranked"))

Artifact version:
v3_scorecard

Primary segment:
Non-financial

Features used by model:
['earnings_risk', 'structural_distress_risk', 'operating_cashflow_risk', 'liquidity_risk', 'debt_service_risk', 'leverage_risk']

Cluster labels:
{4: '1 - Strong relative credit profile', 1: '2 - Good credit profile', 2: '3 - Leveraged / elevated risk profile', 0: '4 - Weak credit profile', 3: '5 - Distressed / near-default proxy'}

Risk rank:
{4: 1, 1: 2, 2: 3, 0: 4, 3: 5}

Cluster sizes:


,cluster,issuer_years,issuers
0,0,2714,1116
1,1,2048,874
2,2,3418,1206
3,3,2354,1085
4,4,1640,639



Cluster profile:


,scorecard_credit_score,structural_distress_risk,earnings_risk,operating_cashflow_risk,liquidity_risk,leverage_risk,debt_service_risk,liabilities_risk,debt_load_risk,equity_buffer_risk,cash_buffer_risk,current_liquidity_risk,quick_liquidity_risk,profitability_risk,cashflow_risk,cfo_to_debt_risk,coverage_risk,fcf_risk,debt_repayment_risk,ebitda_margin_risk,debt_to_ebitda_risk,net_debt_to_ebitda_risk,ebitda_coverage_risk,negative_ebitda_flag,negative_equity_flag,liabilities_exceed_assets_flag,log_assets,small_company_flag,mid_company_flag,large_company_flag,total_debt,net_debt,ebitda,liabilities_to_assets,debt_to_assets,debt_to_equity,equity_to_assets,cash_to_assets,net_income_to_assets,cfo_to_assets,revenue_to_assets,current_ratio,quick_ratio,interest_coverage,fcf_to_debt,operating_margin,gross_margin,cfo_to_liabilities,capex_to_revenue,ebitda_margin,debt_to_ebitda,net_debt_to_ebitda,ebitda_interest_coverage,debt_service_risk_legacy
cluster,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,78.527629,0.225017,1.000000,1.000000,0.260315,0.422658,1.000000,0.176787,0.062887,0.209834,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,0.0,0.0,19.366655,0.0,0.0,1.0,3.061650e+07,-3.128500e+06,-21620750.0,0.414911,0.140877,0.253364,0.513608,0.178877,-0.152602,-0.084680,0.324600,2.975381,1.515642,-14.177290,-0.972332,-0.327905,0.352504,-0.246559,0.025751,-0.266930,8.340862,5.211459,-11.950707,1.000000
1,31.944571,0.235404,0.380970,0.290314,0.264219,0.193328,0.576763,0.241922,0.098969,0.236765,0.337177,0.154070,0.257285,0.380970,0.420704,0.078060,0.828157,0.00000,0.000000,0.526668,0.204806,0.000000,0.775650,0.0,0.0,0.0,21.452476,0.0,0.0,1.0,3.021525e+08,6.436700e+07,139436500.0,0.457249,0.164330,0.327429,0.496103,0.101109,0.042854,0.094824,0.601971,2.191861,1.178394,2.724638,0.410472,0.095949,0.361872,0.208579,0.022125,0.139333,2.024032,0.710996,5.262654,0.595852
2,60.254576,0.595627,0.501852,0.609693,0.671487,0.576082,0.759662,0.615297,0.415515,0.572180,0.784827,0.691215,0.828575,0.501852,0.540404,0.720029,0.860842,0.54391,0.566573,0.418278,0.804981,0.824031,0.847918,0.0,0.0,0.0,22.500720,0.0,0.0,1.0,1.839500e+09,1.411096e+09,400404500.0,0.699943,0.370085,1.080044,0.278083,0.036200,0.024722,0.064899,0.414955,1.117570,0.464281,2.358570,0.128045,0.118983,0.316486,0.095234,0.025416,0.182689,5.024903,4.296124,3.889554,0.726338
3,90.203261,0.877925,1.000000,1.000000,0.758301,0.785994,1.000000,0.913472,0.409703,0.927609,0.481810,0.752608,0.870030,1.000000,1.000000,1.000000,1.000000,1.00000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0,0.0,0.0,19.158992,0.0,0.0,1.0,5.940900e+07,2.617809e+07,-9410927.0,0.893757,0.366307,0.271541,0.047054,0.080138,-0.173378,-0.068558,0.516452,0.994785,0.412462,-3.604414,-0.328566,-0.244551,0.277608,-0.085840,0.012766,-0.191031,11.216977,9.613792,-2.721847,1.000000
4,21.236755,0.331364,0.154353,0.207244,0.377506,0.218323,0.108311,0.321549,0.079934,0.336115,0.430949,0.358071,0.513642,0.154353,0.323700,0.000000,0.000000,0.00000,0.000000,0.431171,0.036720,0.000000,0.113917,0.0,0.0,0.0,21.850772,0.0,0.0,1.0,4.444425e+08,7.403450e+07,447245000.0,0.509007,0.151957,0.317343,0.431525,0.087512,0.076847,0.119075,0.741405,1.783857,0.857948,14.273972,0.596364,0.142820,0.366556,0.237438,0.023953,0.177532,1.183598,0.370943,17.835574,0.073317



Cluster profile ranked:


,financial_flag,cluster,issuer_years,issuers,median_log_assets,median_liabilities_to_assets,median_debt_to_assets,median_debt_to_equity,median_equity_to_assets,median_cash_to_assets,median_net_income_to_assets,median_cfo_to_assets,median_revenue_to_assets,median_current_ratio,median_quick_ratio,median_interest_coverage,median_fcf_to_debt,median_operating_margin,median_gross_margin,median_ebitda_margin,median_debt_to_ebitda,median_net_debt_to_ebitda,median_ebitda_interest_coverage,median_leverage_risk,median_liquidity_risk,median_earnings_risk,median_operating_cashflow_risk,median_debt_service_risk,median_structural_distress_risk,median_scorecard_credit_score,rating_style_rank,rating_style_label
4,Non-financial,4,1640,639,21.850772,0.509007,0.151957,0.317343,0.431525,0.087512,0.076847,0.119075,0.741405,1.783857,0.857948,14.273972,0.596364,0.142820,0.366556,0.177532,1.183598,0.370943,17.835574,0.218323,0.377506,0.154353,0.207244,0.108311,0.331364,21.236755,1,1 - Strong relative credit profile
1,Non-financial,1,2048,874,21.452476,0.457249,0.164330,0.327429,0.496103,0.101109,0.042854,0.094824,0.601971,2.191861,1.178394,2.724638,0.410472,0.095949,0.361872,0.139333,2.024032,0.710996,5.262654,0.193328,0.264219,0.380970,0.290314,0.576763,0.235404,31.944571,2,2 - Good credit profile
2,Non-financial,2,3418,1206,22.500720,0.699943,0.370085,1.080044,0.278083,0.036200,0.024722,0.064899,0.414955,1.117570,0.464281,2.358570,0.128045,0.118983,0.316486,0.182689,5.024903,4.296124,3.889554,0.576082,0.671487,0.501852,0.609693,0.759662,0.595627,60.254576,3,3 - Leveraged / elevated risk profile
0,Non-financial,0,2714,1116,19.366655,0.414911,0.140877,0.253364,0.513608,0.178877,-0.152602,-0.084680,0.324600,2.975381,1.515642,-14.177290,-0.972332,-0.327905,0.352504,-0.266930,8.340862,5.211459,-11.950707,0.422658,0.260315,1.000000,1.000000,1.000000,0.225017,78.527629,4,4 - Weak credit profile
3,Non-financial,3,2354,1085,19.158992,0.893757,0.366307,0.271541,0.047054,0.080138,-0.173378,-0.068558,0.516452,0.994785,0.412462,-3.604414,-0.328566,-0.244551,0.277608,-0.191031,11.216977,9.613792,-2.721847,0.785994,0.758301,1.000000,1.000000,1.000000,0.877925,90.203261,5,5 - Distressed / near-default proxy


## Final notebook conclusion and handoff to Notebook 04

This notebook completes the main applied workflow of the Corporate Credit Clustering Tool. The project has now moved from model construction, then to alternative-model challenges, to real company scoring and reporting. I applied the outcome of the clustering exercise — the saved KMeans artifact — to EEEC 2025 financial data, and the result was interpreted through cluster labels, scorecard diagnostics, affinity, scenario analysis, guardrails, and exported Excel/PDF reports.

For this reason, Notebook 03 is the logical end of the main project communication. It demonstrates the practical business application of the model: a company can be scored against the public-company benchmark, and the result can be explained in a structured, professional format.

Notebook 04 remains part of the project, but it should be read as an optional technical appendix. It explains the EDGAR data-acquisition workflow and the use of the `edgartools` library to collect and update public-company financial data from SEC EDGAR. This is important because the quality of the model depends on the quality and reproducibility of the underlying data pipeline. However, the actual model application and business outcome are already completed in Notebook 03.

I first discovered and used the `edgartools` library in my earlier [Data Science final project](https://github.com/Kamend1/data_science_final_project), and in this project, I extended its role significantly. SEC EDGAR is a free and highly valuable source of US public-company financial data, provided it is accessed responsibly and within traffic rules. Notebook 04, therefore, documents the data-engineering foundation behind the model and provides a path for future updates and extensions.

In short, Notebook 03 closes the main credit-risk modelling and scoring project. Notebook 04 explains and preserves the data-acquisition machinery that makes future model refreshes possible.

The next notebook can be reviewed to understand the original data-acquisition procedure and to use the same workflow as a foundation for designing future EDGAR data collection strategies. After that, I will close the full project with a final summary and my overall conclusions after completing this exercise.

## Full Project Summary and Final Conclusion

When I started this project, the original idea was different from the final result. My initial plan was to collect official corporate credit ratings from Moody’s, S&P, and Fitch, combine them with public financial statement data, perform feature engineering, and train a supervised classification model that could predict rating categories. That idea was natural from a credit-analysis perspective because official ratings provide labels, and supervised machine learning usually needs labels.

However, this original idea ran into a very practical problem: credit ratings are not freely available in a broad, structured, machine-readable way. Rating datasets are valuable commercial products, usually available through platforms such as Bloomberg, FactSet, or similar professional databases. Without reliable labeled data, the original supervised-learning approach became unrealistic within the scope of this project.

This was the moment when the project changed direction. Instead of trying to predict official ratings directly, I started thinking about whether I could create a relative credit-risk segmentation without labels. The clustering lecture gave me the idea to reframe the problem as an unsupervised learning task. The question became: if I cannot train a model on official rating labels, can I still use public financial data and credit-analysis logic to group companies into meaningful credit-risk profiles?

This was both exciting and uncomfortable. It was exciting because the business problem is real. Many companies do not have official credit ratings, but there is still a need to understand their relative credit strength. It was uncomfortable because I was not able to find a directly comparable modelling exercise that applies unsupervised clustering to corporate credit-profile grouping in exactly this way. The available reference framework came mostly from standard financial statement analysis, corporate finance, credit analysis, and general unsupervised machine learning methodology. More than once, I questioned whether I was doing something meaningful or whether I was forcing a machine learning algorithm onto a problem where it did not belong.

In the end, I decided to continue because the financial logic behind the project remained defensible. Credit analysis already works by comparing companies across leverage, liquidity, profitability, cash-flow generation, debt-service capacity, and structural balance-sheet strength. The model does not invent these dimensions. It translates them into a structured feature space and then uses KMeans clustering to group similar financial-risk profiles.

The final project became a domain-guided unsupervised credit-risk benchmarking tool. It is not a formal credit rating model. It is not a probability-of-default model. It is not a regulatory credit assessment. It is a model-relative benchmarking framework that groups non-financial public-company issuer-year observations into five credit-risk buckets based on engineered financial statement features.

The final model uses six bounded domain-level risk features:

| Feature                    | Meaning                                                                       |
| -------------------------- | ----------------------------------------------------------------------------- |
| `structural_distress_risk` | Balance-sheet vulnerability and equity/liability pressure.                    |
| `earnings_risk`            | Profitability weakness.                                                       |
| `operating_cashflow_risk`  | Operating cash-flow weakness relative to assets and debt.                     |
| `liquidity_risk`           | Current liquidity, quick liquidity, cash buffer, and debt repayment capacity. |
| `leverage_risk`            | Balance-sheet leverage and EBITDA-relative debt pressure.                     |
| `debt_service_risk`        | Interest coverage, FCF/debt, debt/EBITDA, and EBITDA coverage pressure.       |

These features are bounded between `0` and `1`, where higher values represent higher financial risk. I deliberately did not cluster on raw accounting values because raw ratios have different scales, directions, and outlier behavior. KMeans is distance-based, so the feature space has to be financially meaningful before clustering is applied.

The final model creates five ranked credit-risk groups:

| Risk rank | Label                             |
| --------: | --------------------------------- |
|         1 | Strong relative credit profile    |
|         2 | Good credit profile               |
|         3 | Leveraged / elevated risk profile |
|         4 | Weak credit profile               |
|         5 | Distressed / near-default proxy   |

The cluster numbers themselves are arbitrary. The business interpretation comes from ranking the clusters after training by their median scorecard credit score and reviewing the financial profile of each cluster. This profiling step was essential because KMeans does not know what “strong” or “weak” credit quality means. It only groups observations by distance. The financial interpretation has to be added afterwards through credit-analysis logic.

A major decision was to keep the model focused on non-financial companies. I intentionally excluded financial and real estate companies from the clustering scope because their balance sheets and credit-risk drivers are different. Banks, insurers, and financial institutions require capital adequacy, asset quality, funding structure, regulatory capital, and liquidity metrics that are not comparable with industrial-company leverage and EBITDA ratios. Real estate companies also require more specialized asset-value, occupancy, rental-income, refinancing, and property-level analysis. They remain a logical future extension, but not part of this version.

The project then evolved into a complete workflow rather than only a model. Notebook 01 builds the main KMeans model. Notebook 02 challenges the modelling decision through Agglomerative Clustering and DBSCAN. Agglomerative Clustering was useful as a hierarchical robustness check, while DBSCAN was useful as a density and outlier diagnostic. The comparison did not give enough reason to replace KMeans. KMeans remained preferable because it provides a stable five-level credit-risk ladder, centroid-based distance diagnostics, and direct compatibility with scoring new company profiles.

Notebook 03 applies the saved model artifact to real company data. This is where the project becomes practical. The notebook loads the trained KMeans artifact, accepts manually entered or CSV-based financial inputs, calculates the same engineered credit-risk features, assigns the company to one of the trained clusters, calculates distances and affinities, applies warning flags and guardrails, runs scenario analysis, and produces Excel and PDF reports.

This is important because the project is not only an academic clustering experiment. It becomes a practical scoring workflow. A company can be compared against a broad public-company benchmark, and the result can be explained in a structured way.

The business applications are real. A business owner could use such a tool to understand how their company might look from a credit-risk perspective before talking to a bank. A bank or lender could use it as an additional financial-risk metric or screening layer, not as a substitute for underwriting. Management teams could use it to monitor customer credit risk, especially when extending trade credit. Procurement or finance departments could also use it to assess the financial stability of key suppliers, at least as an early-warning or benchmarking tool. Investors, consultants, and analysts could use it as a first-pass diagnostic before deeper qualitative review.

One practical test in Notebook 03 used Eastern European Electric Company B.V. (`EEEC`) 2025 financial data. The model assigned the company to the `Leveraged / elevated risk profile` bucket. This was directionally consistent with the company’s actual speculative-grade external ratings, while still respecting the important limitation that my model is not calibrated to Moody’s or Fitch methodology. The model result was driven mainly by leverage and debt-service pressure, while scenario analysis showed how different stresses could affect the scorecard, guardrails, and risk interpretation.

The scenario analysis is still basic, but it demonstrates another important direction of development. A future version could include more refined stress testing: interest-rate shocks, refinancing stress, customer loss, margin compression, working-capital pressure, macroeconomic scenarios, sector-specific scenarios, and multi-year projections. Even in its current form, the scenario section shows that the model can be used not only for a static score, but also for sensitivity analysis around a company’s financial profile.

The reporting output is also important. The project produces CSV files, an Excel workbook, and a professional PDF report. This matters because machine learning work should not stop at a notebook table. If the result is meant to support business users, the output has to be readable, structured, and properly caveated. The PDF and Excel outputs make the model easier to review, explain, and share.

The limitations are serious and should not be hidden. The model is trained on SEC public-company data, which means it may not fully represent private companies, SMEs, Bulgarian companies, or IFRS-only companies. It does not include official rating labels or observed default outcomes. It does not include management quality, ownership support, refinancing access, sector outlook, country risk, customer concentration, collateral, or covenant structure. It is a point-in-time financial-profile model, not a through-the-cycle rating methodology.

At the same time, these limitations do not make the model useless. They define how it should be used. The correct use is as a transparent, reproducible, financially interpretable, model-relative credit-risk benchmark. It is an analyst-support tool, not a replacement for professional credit judgment.

Looking back, the biggest value of the project for me is that it connected my professional credit-analysis background with actual machine learning implementation. The project started from a familiar business problem, hit a real data limitation, changed methodology, and still produced a working model with direct practical application. That evolution itself became one of the most important learning outcomes.

I truly hope that the project review validates the scientific and methodological logic behind the idea. For me, that would be a major success. Not because the model is perfect, and not because I claim to have created an industry-standard credit rating system. I clearly have not. The success would be in showing that the idea is logically structured, financially interpretable, technically implemented, honestly limited, and useful enough to deserve further development.

As an outcome of the Machine Learning course at SoftUni, this project represents much more than a course assignment for me. It is a first serious version of something I have wanted to build for many years: a practical bridge between financial statement analysis, credit-risk thinking, and machine learning. It still needs refinement, validation, and future development, but I believe the foundation is real.


## References

### Credit ratings, credit risk, and financial statement analysis

Altman, E. I. (1968). Financial ratios, discriminant analysis and the prediction of corporate bankruptcy. *The Journal of Finance, 23*(4), 589–609.
https://doi.org/10.1111/j.1540-6261.1968.tb00843.x

Damodaran, A. (2012). *Investment valuation: Tools and techniques for determining the value of any asset* (3rd ed.). John Wiley & Sons.

Moody’s Investors Service. (n.d.). *Understanding ratings*. Moody’s. Retrieved June 16, 2026, from:
https://www.moodys.com/web/en/us/solutions/ratings/understanding-ratings.html

Ohlson, J. A. (1980). Financial ratios and the probabilistic prediction of bankruptcy. *Journal of Accounting Research, 18*(1), 109–131.
https://doi.org/10.2307/2490395

S&P Global Ratings. (n.d.). *Ratings definitions*. S&P Global Ratings. Retrieved June 16, 2026, from:
https://www.spglobal.com/ratings/en/about/intro-to-credit-ratings

Fitch Ratings. (n.d.). *Definitions of ratings and other forms of opinion*. Fitch Ratings. Retrieved June 16, 2026, from:
https://www.fitchratings.com/products/rating-definitions

---

### Machine learning, clustering, and unsupervised model evaluation

Géron, A. (2019). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow: Concepts, Tools, and Techniques to Build Intelligent Systems* (2nd ed.). O’Reilly Media.

---

### SEC EDGAR data and public-company financial filings

U.S. Securities and Exchange Commission. (n.d.). *EDGAR application programming interfaces*. SEC.gov. Retrieved June 16, 2026, from:
https://www.sec.gov/search-filings/edgar-application-programming-interfaces

U.S. Securities and Exchange Commission. (n.d.). *SEC EDGAR company facts API and bulk XBRL data*. SEC.gov. Retrieved June 16, 2026, from:
https://www.sec.gov/Archives/edgar/daily-index/xbrl/companyfacts.zip

U.S. Securities and Exchange Commission. (n.d.). *SEC EDGAR submissions bulk data*. SEC.gov. Retrieved June 16, 2026, from:
https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip

Bommarito, M. J. II, Katz, D. M., & Detterman, E. M. (2018). *OpenEDGAR: Open source software for SEC EDGAR analysis*. arXiv.
https://arxiv.org/abs/1806.04973

---

### Python libraries, machine learning implementation, and data processing tools

EdgarTools. (n.d.). *EdgarTools: Python library for SEC data analysis*. Read the Docs. Retrieved June 16, 2026, from:
https://edgartools.readthedocs.io/

EdgarTools. (n.d.). *EdgarTools — Python library for SEC EDGAR filings*. PyPI. Retrieved June 16, 2026, from:
https://pypi.org/project/edgartools/